In [1]:
# ============================================================
# AlpacaEval fixed-pool Best-of-N proxy migration analysis
# Uses precomputed RM caches only:
#   - ArmoRM
#   - Skywork RM
#   - Tulu3 RM
#
# Input pool:
# /kaggle/input/datasets/pradeep01223/alpacaeval-500x256-llama8b/pool_candidates_merged_300.jsonl
#
# RM score caches:
# /kaggle/input/notebooks/rjjack0112/llama-alpacaeval-armo-score/armorm_cache_128k.csv
# /kaggle/input/notebooks/rjjack0112/llama-alpacaeval-skywork-score/skywork_alpacaeval_scored.csv
# /kaggle/input/notebooks/mrsstrange012/llama-alpacaeval-tulu3-score/tulu3rm_alpacaeval_scored.csv
# ============================================================

import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "sentence-transformers", "textstat", "scipy", "tqdm"
], check=True)

from tqdm.auto import tqdm
from scipy.stats import ttest_rel
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import textstat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.0 MB/s eta 0:00:00


In [2]:
# =========================
# Paths
# =========================
POOL_PATH = Path("/kaggle/input/datasets/pradeep01223/alpacaeval-500x256-llama8b/pool_candidates_merged_300.jsonl")

ARMO_PATH = Path("/kaggle/input/notebooks/rjjack0112/llama-alpacaeval-armo-score/armorm_cache_128k.csv")
SKY_PATH  = Path("/kaggle/input/notebooks/rjjack0112/llama-alpacaeval-skywork-score/skywork_alpacaeval_scored.csv")
TULU_PATH = Path("/kaggle/input/notebooks/mrsstrange012/llama-alpacaeval-tulu3-score/tulu3rm_alpacaeval_scored.csv")

OUT_DIR = Path("/kaggle/working/alpacaeval_migration_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_LIST = [1, 4, 16, 64, 256]

print("POOL exists:", POOL_PATH.exists())
print("ARMO exists:", ARMO_PATH.exists())
print("SKY exists :", SKY_PATH.exists())
print("TULU exists:", TULU_PATH.exists())

POOL exists: True
ARMO exists: True
SKY exists : True
TULU exists: True


In [3]:

# =========================
# Helper metrics
# =========================
KEYWORD_SET = {
    "effective","important","key","great","best","helpful","excellent",
    "significant","valuable","essential","improve","benefit","result",
    "optimize","performance","crucial","critical","powerful","robust",
    "strong","useful","efficient","successful","positive","major",
    "notable","remarkable","outstanding","superior","ideal","optimal",
    "productive","impactful","meaningful","achieve","enhance","boost",
    "increase","maximize","support","enable","ensure","provide","offer",
}

def keyword_density(text):
    words = re.findall(r"\b\w+\b", str(text).lower())
    return sum(1 for w in words if w in KEYWORD_SET) / len(words) if words else 0.0

def response_length(text):
    return len(str(text).split())

def repetition_score(text):
    tokens = str(text).lower().split()
    if len(tokens) < 2:
        return 0.0
    bigrams = list(zip(tokens, tokens[1:]))
    return 1.0 - len(set(bigrams)) / len(bigrams)

def flesch_score(text):
    try:
        return float(textstat.flesch_reading_ease(str(text)))
    except Exception:
        return 50.0

In [4]:
# =========================
# Load STS model
# =========================
sts_model = SentenceTransformer("all-MiniLM-L6-v2")

def sts_score(text, reference):
    e1 = sts_model.encode([str(text)])[0]
    e2 = sts_model.encode([str(reference)])[0]
    return float(1.0 - cosine(e1, e2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# =========================
# Load pool
# =========================
pool_records = []
with open(POOL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pid = int(rec["prompt_id"])
        cands = rec["candidates"]
        pool_records.append({
            "prompt_id": pid,
            "instruction": rec["instruction"],
            "candidates": cands,
            "reference": cands[0],  # N=1 baseline reference, consistent with prior notebooks
            "n_candidates": len(cands),
        })

pool_df = pd.DataFrame([{
    "prompt_id": r["prompt_id"],
    "instruction": r["instruction"],
    "reference": r["reference"],
    "n_candidates": r["n_candidates"],
} for r in pool_records])

pool_map = {r["prompt_id"]: r for r in pool_records}

print(f"Loaded {len(pool_records)} prompts.")
print("Unique candidate counts:", sorted(set(r["n_candidates"] for r in pool_records))[:10])

Loaded 500 prompts.
Unique candidate counts: [256]


In [6]:
# =========================
# RM cache loaders
# =========================
def load_armorm_cache(path: Path):
    df = pd.read_csv(path)
    rename_map = {}
    if "promptid" in df.columns:
        rename_map["promptid"] = "prompt_id"
    if "candidx" in df.columns:
        rename_map["candidx"] = "cand_idx"
    if "cand_idx" in df.columns:
        rename_map["cand_idx"] = "cand_idx"
    if "armorm_raw" in df.columns:
        rename_map["armorm_raw"] = "rm_score"
    df = df.rename(columns=rename_map)
    need = {"prompt_id", "cand_idx", "rm_score"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"ArmoRM cache missing columns: {missing}")
    df["prompt_id"] = df["prompt_id"].astype(int)
    df["cand_idx"] = df["cand_idx"].astype(int)
    df["rm_score"] = df["rm_score"].astype(float)
    df["rm_name"] = "armorm"
    return df[["prompt_id", "cand_idx", "rm_score", "rm_name"]]

def load_skywork_cache(path: Path):
    df = pd.read_csv(path)
    rename_map = {}
    if "prompt_id" in df.columns:
        rename_map["prompt_id"] = "prompt_id"
    if "promptid" in df.columns:
        rename_map["promptid"] = "prompt_id"
    if "candidx" in df.columns:
        rename_map["candidx"] = "cand_idx"
    if "cand_idx" in df.columns:
        rename_map["cand_idx"] = "cand_idx"
    if "skywork_raw" in df.columns:
        rename_map["skywork_raw"] = "rm_score"
    elif "score" in df.columns:
        rename_map["score"] = "rm_score"
    df = df.rename(columns=rename_map)
    need = {"prompt_id", "cand_idx", "rm_score"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Skywork cache missing columns: {missing}")
    df["prompt_id"] = df["prompt_id"].astype(int)
    df["cand_idx"] = df["cand_idx"].astype(int)
    df["rm_score"] = df["rm_score"].astype(float)
    df["rm_name"] = "skywork"
    return df[["prompt_id", "cand_idx", "rm_score", "rm_name"]]

def load_tulu3_cache(path: Path):
    df = pd.read_csv(path)
    rename_map = {}
    if "prompt_id" in df.columns:
        rename_map["prompt_id"] = "prompt_id"
    if "promptid" in df.columns:
        rename_map["promptid"] = "prompt_id"
    if "cand_idx" in df.columns:
        rename_map["cand_idx"] = "cand_idx"
    if "candidx" in df.columns:
        rename_map["candidx"] = "cand_idx"
    if "tulu3_raw" in df.columns:
        rename_map["tulu3_raw"] = "rm_score"
    df = df.rename(columns=rename_map)
    need = {"prompt_id", "cand_idx", "rm_score"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Tulu3 cache missing columns: {missing}")
    df["prompt_id"] = df["prompt_id"].astype(int)
    df["cand_idx"] = df["cand_idx"].astype(int)
    df["rm_score"] = df["rm_score"].astype(float)
    df["rm_name"] = "tulu3"
    return df[["prompt_id", "cand_idx", "rm_score", "rm_name"]]

armo_df = load_armorm_cache(ARMO_PATH)
sky_df  = load_skywork_cache(SKY_PATH)
tulu_df = load_tulu3_cache(TULU_PATH)

print("ArmoRM rows:", len(armo_df))
print("Skywork rows:", len(sky_df))
print("Tulu3 rows:", len(tulu_df))

ArmoRM rows: 128000
Skywork rows: 128000
Tulu3 rows: 128000


In [7]:
# =========================
# Build score caches
# =========================
def build_score_cache(df_rm):
    score_cache = {}
    grouped = df_rm.sort_values(["prompt_id", "cand_idx"]).groupby("prompt_id")
    for pid, g in grouped:
        score_cache[int(pid)] = {
            int(row.cand_idx): float(row.rm_score)
            for row in g.itertuples(index=False)
        }
    return score_cache

score_caches = {
    "armorm": build_score_cache(armo_df),
    "skywork": build_score_cache(sky_df),
    "tulu3": build_score_cache(tulu_df),
}

In [8]:
# =========================
# Validation
# =========================
for rm_name, cache in score_caches.items():
    missing_prompts = []
    short_prompts = []
    for rec in pool_records:
        pid = rec["prompt_id"]
        expected_n = rec["n_candidates"]
        if pid not in cache:
            missing_prompts.append(pid)
            continue
        found = len(cache[pid])
        if found < expected_n:
            short_prompts.append((pid, found, expected_n))
    print(f"\n[{rm_name}] prompts in cache:", len(cache))
    print(f"[{rm_name}] missing prompts:", len(missing_prompts))
    print(f"[{rm_name}] short prompts:", len(short_prompts))
    if missing_prompts[:5]:
        print("First missing prompt IDs:", missing_prompts[:5])
    if short_prompts[:5]:
        print("First short prompt examples:", short_prompts[:5])


[armorm] prompts in cache: 500
[armorm] missing prompts: 0
[armorm] short prompts: 0

[skywork] prompts in cache: 500
[skywork] missing prompts: 0
[skywork] short prompts: 0

[tulu3] prompts in cache: 500
[tulu3] missing prompts: 0
[tulu3] short prompts: 0


In [9]:
# =========================
# Main Best-of-N sweep
# =========================
def run_best_of_n_analysis(rm_name, score_cache):
    rows = []

    for rec in tqdm(pool_records, desc=f"{rm_name} N-sweep"):
        pid = rec["prompt_id"]
        instruction = rec["instruction"]
        cands = rec["candidates"]
        reference = rec["reference"]

        if pid not in score_cache:
            continue

        prompt_scores = score_cache[pid]

        valid_pairs = []
        for idx, cand in enumerate(cands):
            if idx in prompt_scores:
                valid_pairs.append((idx, cand, prompt_scores[idx]))

        if not valid_pairs:
            continue

        valid_pairs = sorted(valid_pairs, key=lambda x: x[0])

        for N in N_LIST:
            pool_n = [(idx, cand, sc) for idx, cand, sc in valid_pairs if idx < N]
            if not pool_n:
                continue

            best_idx, best_text, best_score = max(pool_n, key=lambda x: x[2])

            rows.append({
                "rm_name": rm_name,
                "prompt_id": pid,
                "instruction": instruction,
                "N": N,
                "selected_cand_idx": best_idx,
                "rm_score": best_score,
                "sts": sts_score(best_text, reference),
                "flesch": flesch_score(best_text),
                "resp_len": response_length(best_text),
                "kd": keyword_density(best_text),
                "rep_score": repetition_score(best_text),
                "selected_text": best_text,
                "reference_text": reference,
            })

    df = pd.DataFrame(rows)
    return df

results = {}
for rm_name, cache in score_caches.items():
    df_rm = run_best_of_n_analysis(rm_name, cache)
    results[rm_name] = df_rm
    out_path = OUT_DIR / f"selections_{rm_name}.csv"
    df_rm.to_csv(out_path, index=False)
    print(f"Saved {out_path} with {len(df_rm)} rows")

armorm N-sweep:   0%|          | 0/500 [00:00<?, ?it/s]

Saved /kaggle/working/alpacaeval_migration_outputs/selections_armorm.csv with 2500 rows


skywork N-sweep:   0%|          | 0/500 [00:00<?, ?it/s]

Saved /kaggle/working/alpacaeval_migration_outputs/selections_skywork.csv with 2500 rows


tulu3 N-sweep:   0%|          | 0/500 [00:00<?, ?it/s]

Saved /kaggle/working/alpacaeval_migration_outputs/selections_tulu3.csv with 2500 rows


In [10]:
# =========================
# Aggregate summaries
# =========================
def summarize_rm(df_rm):
    agg = df_rm.groupby("N").agg(
        rm_mean=("rm_score", "mean"),
        rm_std=("rm_score", "std"),
        sts_mean=("sts", "mean"),
        sts_std=("sts", "std"),
        flesch_mean=("flesch", "mean"),
        flesch_std=("flesch", "std"),
        resp_len_mean=("resp_len", "mean"),
        resp_len_std=("resp_len", "std"),
        kd_mean=("kd", "mean"),
        kd_std=("kd", "std"),
        rep_mean=("rep_score", "mean"),
        rep_std=("rep_score", "std"),
    ).round(5)

    base_r = float(agg.loc[1, "rm_mean"])
    eps = 1e-9
    agg["ERR"] = ((agg["rm_mean"] - base_r) / (abs(base_r) + eps)).round(5)

    n1 = df_rm[df_rm["N"] == 1].sort_values("prompt_id")
    nmax = df_rm[df_rm["N"] == max(N_LIST)].sort_values("prompt_id")

    merged = n1[["prompt_id", "rm_score", "sts", "flesch", "resp_len", "kd", "rep_score"]].merge(
        nmax[["prompt_id", "rm_score", "sts", "flesch", "resp_len", "kd", "rep_score"]],
        on="prompt_id",
        suffixes=("_n1", "_nmax")
    )

    t_rm = ttest_rel(merged["rm_score_nmax"], merged["rm_score_n1"], nan_policy="omit")
    t_sts = ttest_rel(merged["sts_nmax"], merged["sts_n1"], nan_policy="omit")
    t_flesch = ttest_rel(merged["flesch_nmax"], merged["flesch_n1"], nan_policy="omit")
    t_len = ttest_rel(merged["resp_len_nmax"], merged["resp_len_n1"], nan_policy="omit")
    t_kd = ttest_rel(merged["kd_nmax"], merged["kd_n1"], nan_policy="omit")
    t_rep = ttest_rel(merged["rep_score_nmax"], merged["rep_score_n1"], nan_policy="omit")

    stats_row = pd.DataFrame([{
        "rm_name": df_rm["rm_name"].iloc[0],
        "n_prompts": merged["prompt_id"].nunique(),
        "N_max": max(N_LIST),
        "rm_gain_n1_to_nmax": merged["rm_score_nmax"].mean() - merged["rm_score_n1"].mean(),
        "sts_delta_n1_to_nmax": merged["sts_nmax"].mean() - merged["sts_n1"].mean(),
        "flesch_delta_n1_to_nmax": merged["flesch_nmax"].mean() - merged["flesch_n1"].mean(),
        "resp_len_delta_n1_to_nmax": merged["resp_len_nmax"].mean() - merged["resp_len_n1"].mean(),
        "kd_delta_n1_to_nmax": merged["kd_nmax"].mean() - merged["kd_n1"].mean(),
        "rep_delta_n1_to_nmax": merged["rep_score_nmax"].mean() - merged["rep_score_n1"].mean(),
        "rm_t": t_rm.statistic,
        "rm_p": t_rm.pvalue,
        "sts_t": t_sts.statistic,
        "sts_p": t_sts.pvalue,
        "flesch_t": t_flesch.statistic,
        "flesch_p": t_flesch.pvalue,
        "len_t": t_len.statistic,
        "len_p": t_len.pvalue,
        "kd_t": t_kd.statistic,
        "kd_p": t_kd.pvalue,
        "rep_t": t_rep.statistic,
        "rep_p": t_rep.pvalue,
    }])

    return agg.reset_index(), stats_row, merged

all_stats = []
for rm_name, df_rm in results.items():
    agg_df, stats_df, paired_df = summarize_rm(df_rm)

    agg_path = OUT_DIR / f"{rm_name}_summary_by_N.csv"
    stats_path = OUT_DIR / f"{rm_name}_paired_tests.csv"
    paired_path = OUT_DIR / f"{rm_name}_paired_n1_vs_n256.csv"

    agg_df.to_csv(agg_path, index=False)
    stats_df.to_csv(stats_path, index=False)
    paired_df.to_csv(paired_path, index=False)

    all_stats.append(stats_df)

    print(f"\n=== {rm_name.upper()} ===")
    print(agg_df[["N", "rm_mean", "sts_mean", "flesch_mean", "resp_len_mean", "kd_mean", "ERR"]].to_string(index=False))
    print("\nPaired N=1 vs N=256:")
    print(stats_df.to_string(index=False))

all_stats_df = pd.concat(all_stats, ignore_index=True)
all_stats_df.to_csv(OUT_DIR / "all_rm_paired_tests.csv", index=False)


=== ARMORM ===
  N  rm_mean  sts_mean  flesch_mean  resp_len_mean  kd_mean     ERR
  1  0.04655   1.00000     37.71233        185.018  0.00798 0.00000
  4  0.07957   0.72052     41.52585        181.770  0.00766 0.70934
 16  0.09643   0.67480     42.76076        178.202  0.00799 1.07154
 64  0.10694   0.66596     43.24240        172.688  0.00820 1.29731
256  0.11483   0.65990     42.32667        165.304  0.00773 1.46681

Paired N=1 vs N=256:
rm_name  n_prompts  N_max  rm_gain_n1_to_nmax  sts_delta_n1_to_nmax  flesch_delta_n1_to_nmax  resp_len_delta_n1_to_nmax  kd_delta_n1_to_nmax  rep_delta_n1_to_nmax      rm_t          rm_p      sts_t         sts_p  flesch_t  flesch_p     len_t        len_p      kd_t    kd_p    rep_t        rep_p
 armorm        500    256            0.068283             -0.340096                 4.614341                    -19.714            -0.000255              0.011921 34.205913 6.306372e-133 -36.703535 7.299292e-144  3.852407  0.000132 -9.872324 4.071487e-21 -0.5

In [11]:
# =========================
# Cross-RM combined summary
# =========================
combined_agg = []
for rm_name in results:
    agg_df = pd.read_csv(OUT_DIR / f"{rm_name}_summary_by_N.csv")
    agg_df["rm_name"] = rm_name
    combined_agg.append(agg_df)

combined_agg_df = pd.concat(combined_agg, ignore_index=True)
combined_agg_df.to_csv(OUT_DIR / "combined_summary_by_N.csv", index=False)

print("\nSaved all outputs to:", OUT_DIR)


Saved all outputs to: /kaggle/working/alpacaeval_migration_outputs


In [12]:
# =========================
# Optional compact verdicts
# =========================
def simple_verdict(agg_df):
    a = agg_df.set_index("N")
    rm_gain = a.loc[256, "rm_mean"] - a.loc[1, "rm_mean"]
    sts_delta = a.loc[256, "sts_mean"] - a.loc[1, "sts_mean"]
    err_256 = a.loc[256, "ERR"]
    if rm_gain > 0 and sts_delta < -0.003:
        return f"TYPE III pattern present | rm_gain={rm_gain:.4f} | sts_delta={sts_delta:.4f} | ERR256={err_256:.4f}"
    elif rm_gain > 0 and sts_delta < 0:
        return f"Directional drift only | rm_gain={rm_gain:.4f} | sts_delta={sts_delta:.4f} | ERR256={err_256:.4f}"
    else:
        return f"No clear Type III pattern | rm_gain={rm_gain:.4f} | sts_delta={sts_delta:.4f} | ERR256={err_256:.4f}"

print("\n=== Compact verdicts ===")
for rm_name in results:
    agg_df = pd.read_csv(OUT_DIR / f"{rm_name}_summary_by_N.csv")
    print(rm_name, "->", simple_verdict(agg_df))


=== Compact verdicts ===
armorm -> TYPE III pattern present | rm_gain=0.0683 | sts_delta=-0.3401 | ERR256=1.4668
skywork -> TYPE III pattern present | rm_gain=17.4434 | sts_delta=-0.3395 | ERR256=1.1878
tulu3 -> TYPE III pattern present | rm_gain=4.0366 | sts_delta=-0.3329 | ERR256=1.1910
